# Step 5B — Refine the camera-to-command CNN before autonomous control

This notebook is a **controlled refinement experiment** for the accepted Step-5 model. It does not change CoppeliaSim, the camera format, the CNN architecture, the audited dataset, or the train/validation/test split.

The reason for this refinement is the clean Step-6A live shadow run of maze_navigation_model_v2.h5:

- overall human/model agreement: **90.54%**;
- human forward predicted as left: **72 / 833 = 8.64%**;
- several false-left streaks lasted multiple consecutive frames.

The working hypothesis is that full inverse-frequency class balancing made the minority turn classes disproportionately influential. This notebook tests **only one controlled variable**: the strength of class weighting.

Candidate selection uses **validation data only**. The untouched test set is evaluated exactly once, after a candidate has been selected. The final model is saved separately as maze_navigation_model_v3.h5; v2 is never overwritten.

**Important:** this notebook trains and evaluates models only. It does not connect to CoppeliaSim and cannot actuate the robot.


## 1. Configuration and experimental guardrails

Use the same audited dataset_v2_clean_2383.zip that produced the accepted v2 model. The exact expected class counts are enforced so the experiment cannot silently use the 2386-image working dataset or another archive.

Weight-strength alpha is a power interpolation of the original balanced weights:

- alpha = 1.00 → original full balanced weighting (baseline reproduction);
- alpha = 0.00 → no class weighting;
- intermediate values → progressively moderated weighting.

Every candidate is normalized so the mean sample weight over the training set remains 1.0. This changes relative class emphasis without arbitrarily changing the overall loss scale.


In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intelligent_Robotics_Design_Project"
DATASET_ZIP_NAME = "dataset_v2_clean_2383.zip"
MODEL_FILENAME = "maze_navigation_model_v3.h5"
CANDIDATE_METRICS_FILENAME = "step05b_candidate_validation_metrics.csv"
SELECTED_CONFIG_FILENAME = "step05b_selected_config.json"
TEST_METRICS_FILENAME = "step05b_v3_test_metrics.json"

RANDOM_SEED = 42
IMAGE_SIZE = (50, 50)
BATCH_SIZE = 32
EPOCHS = 30

EXPECTED_CLASS_COUNTS = {"forward": 2137, "left": 121, "right": 125}

CANDIDATE_WEIGHT_ALPHAS = [1.00, 0.75, 0.50, 0.25, 0.00]

# Validation contains only 18 left and 19 right examples, so one example changes
# turn recall by about five percentage points. Use an integer-count guardrail
# rather than a brittle percentage-point threshold.
MAX_ADDITIONAL_TURN_MISSES = 2
# Retained as historical documentation of the first guardrail; selection below
# uses integer turn-miss counts because the turn validation supports are small.
TURN_RECALL_TOLERANCE = 0.10
MACRO_F1_TOLERANCE = 0.05

V2_REFERENCE = {
    "test_accuracy": 0.9022,
    "forward_to_left_errors": 29,
    "forward_support": 321,
    "forward_to_left_rate": 29 / 321,
    "left_recall": 1.0,
    "right_recall": 14 / 19,
}


## 2. Mount Google Drive and import tools

Approve the Google Drive prompt when Colab asks. TensorFlow trains the CNN; scikit-learn provides the deterministic split, class weighting, confusion matrices, and per-class metrics.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()

project_dir = Path(DRIVE_PROJECT_DIR)
zip_path = project_dir / DATASET_ZIP_NAME
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the audited dataset ZIP here first: {zip_path}")

print(f"TensorFlow: {tf.__version__}")
print(f"Dataset ZIP: {zip_path}")


## 3. Extract the audited ZIP safely

The ZIP is extracted to temporary Colab storage. Windows-style backslashes are normalized because the archive was produced on Windows. Unsafe or ambiguous archive paths stop the notebook.


In [ ]:
CLASS_NAMES = ["forward", "left", "right"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

print("MODEL OUTPUT CLASS MAPPING:")
for name, index in CLASS_TO_INDEX.items():
    print(f"  {name} -> {index}")

extract_dir = Path("/content/maze_dataset_step05b")
with zipfile.ZipFile(zip_path) as archive:
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)

    for member in archive.infolist():
        normalized_name = member.filename.replace(chr(92), "/")
        normalized_path = Path(normalized_name)

        if (
            normalized_path.is_absolute()
            or ".." in normalized_path.parts
            or not normalized_path.parts
        ):
            raise ValueError(f"Unsafe ZIP member: {member.filename}")

        destination = extract_dir.joinpath(*normalized_path.parts)
        if member.is_dir() or normalized_name.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)

dataset_dir = extract_dir
while True:
    entries = list(dataset_dir.iterdir())
    if len(entries) == 1 and entries[0].is_dir() and entries[0].name not in CLASS_NAMES:
        dataset_dir = entries[0]
    else:
        break

if {p.name for p in dataset_dir.iterdir()} != set(CLASS_NAMES):
    raise ValueError(
        f"Expected exactly {CLASS_NAMES} in {dataset_dir}; "
        f"found {[p.name for p in dataset_dir.iterdir()]}"
    )
if not all((dataset_dir / name).is_dir() for name in CLASS_NAMES):
    raise ValueError("Every class entry must be a directory.")

print(f"Verified dataset root: {dataset_dir}")


## 4. Validate every image and enforce the 2383-image source of truth

Every file must be a valid RGB JPEG. This notebook deliberately refuses to continue unless the class counts are exactly **2137 forward, 121 left, 125 right**.


In [ ]:
records = []
invalid_files = []
original_shapes = set()

for class_name in CLASS_NAMES:
    class_dir = dataset_dir / class_name
    for path in sorted(class_dir.iterdir()):
        if not path.is_file() or path.suffix.lower() not in {".jpg", ".jpeg"}:
            invalid_files.append(f"{path}: expected a JPEG file")
            continue
        try:
            decoded = tf.io.decode_jpeg(tf.io.read_file(str(path)), channels=3)
            shape = tuple(decoded.shape.as_list())
            if len(shape) != 3 or shape[0] <= 0 or shape[1] <= 0 or shape[2] != 3:
                raise ValueError(f"Invalid RGB shape: {shape}")
            original_shapes.add(shape)
            records.append({"path": str(path), "label": CLASS_TO_INDEX[class_name]})
        except Exception as error:
            invalid_files.append(f"{path}: {error}")

if invalid_files:
    raise ValueError("Dataset validation failed:\n" + "\n".join(invalid_files))

images_df = pd.DataFrame(records, columns=["path", "label"])
counts = images_df["label"].value_counts().reindex(range(3), fill_value=0)
actual_counts = {CLASS_NAMES[i]: int(counts[i]) for i in range(3)}

print("Validated class counts:", actual_counts)
print(f"Total: {len(images_df)}")
print(f"Original image shapes: {sorted(original_shapes)}")

if actual_counts != EXPECTED_CLASS_COUNTS:
    raise ValueError(
        "Wrong dataset for Step 5B. "
        f"Expected {EXPECTED_CLASS_COUNTS}, found {actual_counts}."
    )
if len(images_df) != 2383:
    raise ValueError(f"Expected exactly 2383 images, found {len(images_df)}.")

print("AUDITED DATASET CHECK: PASS")


## 5. Reproduce the accepted deterministic train/validation/test split

The split logic is unchanged from Step 5: 70% training, 15% validation, 15% test using RANDOM_SEED = 42 and stratification.

The test split is created here but **not evaluated or inspected for candidate selection**.


In [ ]:
train_df, held_out_df = train_test_split(
    images_df,
    test_size=0.30,
    stratify=images_df["label"],
    random_state=RANDOM_SEED,
)
val_df, test_df = train_test_split(
    held_out_df,
    test_size=0.50,
    stratify=held_out_df["label"],
    random_state=RANDOM_SEED,
)

for name, frame in [("Train", train_df), ("Validation", val_df), ("Test (sealed)", test_df)]:
    print(f"{name}: {len(frame)} images ({len(frame) / len(images_df):.1%})")
    print(
        frame["label"]
        .value_counts()
        .reindex(range(3), fill_value=0)
        .rename(index=dict(enumerate(CLASS_NAMES)))
        .to_string()
    )

assert set(train_df.path).isdisjoint(val_df.path)
assert set(train_df.path).isdisjoint(test_df.path)
assert set(val_df.path).isdisjoint(test_df.path)

print("SPLIT ISOLATION CHECK: PASS")


## 6. Prepare the image pipeline

Image preprocessing is intentionally unchanged from v2: RGB decode, resize to **50 × 50**, convert to float32, and scale to [0, 1].

Only the training dataset is shuffled.


In [ ]:
def load_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32) / 255.0

def load_example(path, label):
    return load_image(path), label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["path"].to_numpy(),
        frame["label"].to_numpy(dtype=np.int32),
    ))
    dataset = dataset.map(
        load_example,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )
    if training:
        dataset = dataset.shuffle(
            len(frame),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True,
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = make_dataset(val_df, training=False)


## 7. Derive the candidate class weights

The original Step-5 balanced weights are computed first. Each candidate applies balanced_weight raised to alpha, then normalizes the resulting sample weights so their training-set mean remains 1.0.

This makes the candidate series systematic and reproducible rather than a set of hand-picked class weights.


In [ ]:
train_labels = train_df["label"].to_numpy(dtype=np.int32)
train_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))

balanced_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=train_labels,
).astype(np.float64)

def class_weights_for_alpha(alpha):
    raw = np.power(balanced_weights_array, float(alpha))
    mean_sample_weight = float(np.dot(train_counts, raw) / train_counts.sum())
    normalized = raw / mean_sample_weight
    return {i: float(normalized[i]) for i in range(len(CLASS_NAMES))}

weight_rows = []
for alpha in CANDIDATE_WEIGHT_ALPHAS:
    weights = class_weights_for_alpha(alpha)
    row = {"alpha": alpha}
    row.update({name: weights[i] for i, name in enumerate(CLASS_NAMES)})
    row["mean_sample_weight"] = float(
        np.dot(train_counts, np.array([weights[i] for i in range(3)])) / train_counts.sum()
    )
    weight_rows.append(row)

weights_df = pd.DataFrame(weight_rows)
display(weights_df.round(4))
assert np.allclose(weights_df["mean_sample_weight"].to_numpy(), 1.0, atol=1e-10)

print("WEIGHT NORMALIZATION CHECK: PASS")


## 8. Keep the v2 CNN architecture unchanged

Only class-weight strength is being tested. The convolutional architecture, optimizer, loss function, dropout, image size, seed, and early-stopping policy remain the same as Step 5.


In [ ]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

probe_model = build_model()
probe_model.summary()
del probe_model
tf.keras.backend.clear_session()


## 9. Define validation metrics

The principal failure metric is **forward → left rate**, because that was the live Step-6A failure mode.

We also retain overall accuracy, macro-F1, and per-class recall so a candidate cannot look better merely by refusing to recognize genuine turns.


In [ ]:
def prediction_metrics(frame, probabilities):
    labels = frame["label"].to_numpy(dtype=np.int32)
    predictions = np.argmax(probabilities, axis=1)
    matrix = confusion_matrix(labels, predictions, labels=np.arange(3))
    precision, recall, f1, support = precision_recall_fscore_support(
        labels,
        predictions,
        labels=np.arange(3),
        zero_division=0,
    )

    forward_support = int(matrix[0].sum())
    forward_to_left_errors = int(matrix[0, 1])

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(np.mean(f1)),
        "forward_to_left_errors": forward_to_left_errors,
        "forward_support": forward_support,
        "forward_to_left_rate": (
            float(forward_to_left_errors / forward_support)
            if forward_support else 0.0
        ),
        "forward_precision": float(precision[0]),
        "forward_recall": float(recall[0]),
        "left_precision": float(precision[1]),
        "left_recall": float(recall[1]),
        "left_support": int(support[1]),
        "left_correct": int(matrix[1, 1]),
        "right_precision": float(precision[2]),
        "right_recall": float(recall[2]),
        "right_support": int(support[2]),
        "right_correct": int(matrix[2, 2]),
        "confusion_matrix": matrix,
    }


## 10. Train all candidates — validation data only

Each candidate starts from the same deterministic random seed and the same architecture. The training dataset is recreated for every candidate so the comparison is controlled.

This cell can take several minutes. It intentionally does **not** touch the test set.


In [ ]:
candidate_rows = []
candidate_weights = {}
candidate_histories = {}
candidate_confusions = {}

for alpha in CANDIDATE_WEIGHT_ALPHAS:
    label = f"alpha_{alpha:.2f}"
    print("\n" + "=" * 72)
    print(f"TRAINING CANDIDATE: {label}")
    print("=" * 72)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_SEED)

    model = build_model()
    train_ds = make_dataset(train_df, training=True)
    weights = class_weights_for_alpha(alpha)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=weights,
        callbacks=[early_stopping],
        verbose=2,
    )

    val_probabilities = model.predict(val_ds, verbose=0)
    metrics = prediction_metrics(val_df, val_probabilities)

    candidate_rows.append({
        "candidate": label,
        "alpha": float(alpha),
        "epochs_ran": len(history.history["loss"]),
        "val_accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "forward_to_left_errors": metrics["forward_to_left_errors"],
        "forward_support": metrics["forward_support"],
        "forward_to_left_rate": metrics["forward_to_left_rate"],
        "forward_recall": metrics["forward_recall"],
        "left_precision": metrics["left_precision"],
        "left_recall": metrics["left_recall"],
        "left_support": metrics["left_support"],
        "left_correct": metrics["left_correct"],
        "right_precision": metrics["right_precision"],
        "right_recall": metrics["right_recall"],
        "right_support": metrics["right_support"],
        "right_correct": metrics["right_correct"],
        "weight_forward": weights[0],
        "weight_left": weights[1],
        "weight_right": weights[2],
    })

    candidate_weights[label] = [np.array(weight, copy=True) for weight in model.get_weights()]
    candidate_histories[label] = {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    }
    candidate_confusions[label] = metrics["confusion_matrix"].copy()

    print(
        f"{label}: val_accuracy={metrics['accuracy']:.2%}, "
        f"macro_f1={metrics['macro_f1']:.4f}, "
        f"forward->left={metrics['forward_to_left_errors']}/{metrics['forward_support']} "
        f"({metrics['forward_to_left_rate']:.2%}), "
        f"left_recall={metrics['left_recall']:.2%}, "
        f"right_recall={metrics['right_recall']:.2%}"
    )

results_df = pd.DataFrame(candidate_rows).sort_values("alpha", ascending=False).reset_index(drop=True)

project_dir.mkdir(parents=True, exist_ok=True)
candidate_metrics_path = project_dir / CANDIDATE_METRICS_FILENAME
results_df.to_csv(candidate_metrics_path, index=False)

print(f"\nSaved validation-only candidate metrics: {candidate_metrics_path}")
display(
    results_df[
        [
            "candidate",
            "alpha",
            "val_accuracy",
            "macro_f1",
            "forward_to_left_errors",
            "forward_to_left_rate",
            "left_precision",
            "left_recall",
            "left_correct",
            "left_support",
            "right_precision",
            "right_recall",
            "right_correct",
            "right_support",
        ]
    ].style.format({
        "val_accuracy": "{:.2%}",
        "macro_f1": "{:.4f}",
        "forward_to_left_rate": "{:.2%}",
        "left_precision": "{:.2%}",
        "left_recall": "{:.2%}",
        "right_precision": "{:.2%}",
        "right_recall": "{:.2%}",
    })
)


## 11. Select the v3 candidate using validation data only

The alpha=1.00 run is the baseline.

The first Run-all exposed an important small-sample issue in the original guardrail: the validation split contains only **18 left** and **19 right** images, so one left image changes recall by about **5.56 percentage points**. The original 10-point recall rule rejected alpha=0.75 by only one image even though that candidate sharply reduced forward → left errors and improved validation accuracy and macro-F1.

The corrected guardrail therefore uses **integer turn mistakes**, which is more interpretable at this sample size. A candidate is eligible only if:

- it makes at most **2 additional left mistakes** versus the alpha=1.00 baseline;
- it makes at most **2 additional right mistakes** versus the alpha=1.00 baseline;
- macro-F1 is no more than 0.05 below baseline.

Among eligible candidates, selection first minimizes **forward → left rate**, then maximizes macro-F1, then validation accuracy.

This revision still uses **validation data only**. The held-out test set remains sealed until after a candidate is selected. If no moderated candidate improves forward → left rate under these count-based guardrails, the notebook stops before test evaluation.


In [ ]:
baseline = results_df.loc[np.isclose(results_df["alpha"], 1.0)].iloc[0]

left_correct_floor = max(
    0,
    int(baseline["left_correct"]) - MAX_ADDITIONAL_TURN_MISSES,
)
right_correct_floor = max(
    0,
    int(baseline["right_correct"]) - MAX_ADDITIONAL_TURN_MISSES,
)
macro_f1_floor = max(0.0, float(baseline["macro_f1"]) - MACRO_F1_TOLERANCE)

eligible_df = results_df[
    (results_df["left_correct"] >= left_correct_floor)
    & (results_df["right_correct"] >= right_correct_floor)
    & (results_df["macro_f1"] >= macro_f1_floor)
].copy()

if eligible_df.empty:
    raise RuntimeError("No candidate satisfies the validation guardrails.")

ranked_df = eligible_df.sort_values(
    ["forward_to_left_rate", "macro_f1", "val_accuracy", "alpha"],
    ascending=[True, False, False, False],
).reset_index(drop=True)

selected = ranked_df.iloc[0]
selected_label = str(selected["candidate"])

print("Baseline validation metrics:")
display(baseline.to_frame("baseline").T)
print(
    "\nCount-based turn guardrails: "
    f"left_correct >= {left_correct_floor}/{int(baseline['left_support'])}; "
    f"right_correct >= {right_correct_floor}/{int(baseline['right_support'])}"
)
print("\nEligible candidates ranked for selection:")
display(ranked_df)

if float(selected["forward_to_left_rate"]) >= float(baseline["forward_to_left_rate"]) - 1e-12:
    raise RuntimeError(
        "No moderated candidate reduced validation forward->left errors "
        "without exceeding the turn-performance guardrails. "
        "STOP here; do not tune against the test set."
    )

print(f"\nSELECTED CANDIDATE: {selected_label}")
print(
    f"forward->left rate improved from "
    f"{float(baseline['forward_to_left_rate']):.2%} "
    f"to {float(selected['forward_to_left_rate']):.2%} on validation."
)
print(
    "Turn validation counts: "
    f"left {int(selected['left_correct'])}/{int(selected['left_support'])} "
    f"(baseline {int(baseline['left_correct'])}/{int(baseline['left_support'])}); "
    f"right {int(selected['right_correct'])}/{int(selected['right_support'])} "
    f"(baseline {int(baseline['right_correct'])}/{int(baseline['right_support'])})."
)

print("\nBaseline confusion matrix (validation):")
display(pd.DataFrame(
    candidate_confusions[str(baseline["candidate"])],
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
))
print("\nSelected confusion matrix (validation):")
display(pd.DataFrame(
    candidate_confusions[selected_label],
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
))

selected_config = {
    "selected_candidate": selected_label,
    "selected_alpha": float(selected["alpha"]),
    "random_seed": RANDOM_SEED,
    "image_size": list(IMAGE_SIZE),
    "dataset_zip": DATASET_ZIP_NAME,
    "dataset_counts": EXPECTED_CLASS_COUNTS,
    "validation_baseline_forward_to_left_rate": float(baseline["forward_to_left_rate"]),
    "validation_selected_forward_to_left_rate": float(selected["forward_to_left_rate"]),
    "validation_selected_left_recall": float(selected["left_recall"]),
    "validation_selected_right_recall": float(selected["right_recall"]),
    "validation_selected_left_correct": int(selected["left_correct"]),
    "validation_selected_right_correct": int(selected["right_correct"]),
    "selection_policy": {
        "max_additional_turn_misses": MAX_ADDITIONAL_TURN_MISSES,
        "macro_f1_tolerance": MACRO_F1_TOLERANCE,
        "ranking": [
            "min forward_to_left_rate",
            "max macro_f1",
            "max val_accuracy",
        ],
    },
}
selected_config_path = project_dir / SELECTED_CONFIG_FILENAME
selected_config_path.write_text(json.dumps(selected_config, indent=2))
print(f"Saved selected configuration: {selected_config_path}")


## 12. Visualize the validation trade-off

A good candidate should move left on this plot, meaning fewer false-left predictions, without collapsing downward in macro-F1.


In [ ]:
plt.figure(figsize=(8, 5))
for _, row in results_df.iterrows():
    x = 100.0 * float(row["forward_to_left_rate"])
    y = float(row["macro_f1"])
    plt.scatter(x, y, s=70)
    plt.annotate(
        row["candidate"],
        (x, y),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel("Validation forward → left error rate (%) — lower is better")
plt.ylabel("Validation macro-F1 — higher is better")
plt.title("Step 5B class-weight trade-off")
plt.grid(alpha=0.3)
plt.show()


## 13. Restore the selected candidate and unseal the test set

Only now, after selection has finished, is the held-out test set used.

The selected model weights are restored into a fresh instance of the unchanged CNN architecture.


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)

selected_model = build_model()
selected_model.set_weights(candidate_weights[selected_label])

test_ds = make_dataset(test_df, training=False)
test_loss, test_accuracy = selected_model.evaluate(test_ds, verbose=0)
test_probabilities = selected_model.predict(test_ds, verbose=0)
test_metrics = prediction_metrics(test_df, test_probabilities)

test_confusion = test_metrics.pop("confusion_matrix")

print(f"Selected candidate: {selected_label}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")
print(
    f"Test forward->left: "
    f"{test_metrics['forward_to_left_errors']}/{test_metrics['forward_support']} "
    f"({test_metrics['forward_to_left_rate']:.2%})"
)
print(f"Test left recall: {test_metrics['left_recall']:.2%}")
print(f"Test right recall: {test_metrics['right_recall']:.2%}")

print("\nConfusion matrix (rows=true, columns=predicted):")
display(pd.DataFrame(test_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))

print("\nAccepted v2 reference (not used for candidate selection):")
print(
    f"  accuracy={V2_REFERENCE['test_accuracy']:.2%}, "
    f"forward->left="
    f"{V2_REFERENCE['forward_to_left_errors']}/{V2_REFERENCE['forward_support']} "
    f"({V2_REFERENCE['forward_to_left_rate']:.2%}), "
    f"left_recall={V2_REFERENCE['left_recall']:.2%}, "
    f"right_recall={V2_REFERENCE['right_recall']:.2%}"
)


## 14. Save v3 and its evidence to Google Drive

The selected model is saved separately as maze_navigation_model_v3.h5. The existing v2 model is not modified.

The notebook also saves machine-readable JSON and CSV evidence so the refinement decision can be audited later.


In [ ]:
model_path = project_dir / MODEL_FILENAME
selected_model.save(str(model_path))
if not model_path.is_file() or model_path.stat().st_size == 0:
    raise RuntimeError(f"Model save verification failed: {model_path}")

def json_number(value):
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    return value

test_record = {
    "selected_candidate": selected_label,
    "selected_alpha": float(selected["alpha"]),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    **{key: json_number(value) for key, value in test_metrics.items()},
    "confusion_matrix": test_confusion.astype(int).tolist(),
    "v2_reference": V2_REFERENCE,
}
test_metrics_path = project_dir / TEST_METRICS_FILENAME
test_metrics_path.write_text(json.dumps(test_record, indent=2))

print(f"Verified saved v3 HDF5 model: {model_path}")
print(f"File size: {model_path.stat().st_size:,} bytes")
print(f"Validation candidate table: {candidate_metrics_path}")
print(f"Selected configuration: {selected_config_path}")
print(f"Test metrics: {test_metrics_path}")


## 15. Plot the selected candidate's learning curves

These curves are preserved for interpretation. Candidate selection itself has already been completed from the validation metrics above.


In [ ]:
history = candidate_histories[selected_label]
epoch_numbers = range(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric, title in zip(
    axes,
    ["accuracy", "loss"],
    ["Accuracy", "Loss"],
):
    axis.plot(epoch_numbers, history[metric], label="Training")
    axis.plot(epoch_numbers, history[f"val_{metric}"], label="Validation")
    axis.set(
        xlabel="Epoch",
        ylabel=title,
        title=f"Selected v3 candidate: training and validation {title.lower()}",
    )
    axis.legend()
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 16. Demonstrate one held-out prediction

This final demonstration uses the already-unsealed test set and the selected v3 model. It is for interpretation only and does not affect model selection.


In [ ]:
example = test_df.iloc[0]
example_image = load_image(example["path"])
probabilities = selected_model.predict(example_image[None, ...], verbose=0)[0]
predicted_index = int(np.argmax(probabilities))

print(f"True label: {CLASS_NAMES[int(example['label'])]}")
print(f"Predicted label: {CLASS_NAMES[predicted_index]}")
print("Softmax probabilities:")
for name, probability in zip(CLASS_NAMES, probabilities):
    print(f"  {name}: {probability:.6f}")

plt.imshow(example_image.numpy())
plt.title(
    f"True: {CLASS_NAMES[int(example['label'])]} | "
    f"Predicted: {CLASS_NAMES[predicted_index]}"
)
plt.axis("off")
plt.show()


## 17. Preserve the Step-5B outputs

After a successful **Run all**, preserve:

- **step05b_cnn_refinement.ipynb** — the executed notebook with candidate metrics, selection evidence, test metrics, and plots visible;
- **maze_navigation_model_v3.h5** — the selected refined CNN;
- **step05b_candidate_validation_metrics.csv** — validation metrics for every class-weight candidate;
- **step05b_selected_config.json** — the validation-only selection decision;
- **step05b_v3_test_metrics.json** — the one-time held-out test result;
- **dataset_v2_clean_2383.zip** — unchanged source dataset already preserved separately.

Do **not** delete or overwrite maze_navigation_model_v2.h5. The next gate is a new Step-6A live shadow run comparing v3 against human W/A/D driving. Autonomous motor actuation remains disabled until that live validation is acceptable.
